In [1]:
# Basic tools
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Save model
import joblib


In [2]:
import pandas as pd

file_path = "merged_output.csv"   # keep this name, or change if needed

# ✅ Robust loading: skip broken lines + use chunks
chunk_iter = pd.read_csv(
    file_path,
    engine="python",      # safer parser
    on_bad_lines="skip",  # ignore rows with wrong number of columns
    chunksize=200000      # read 200k rows at a time
)

# ✅ Combine all chunks into one DataFrame
df = pd.concat(chunk_iter, ignore_index=True)

df.head()


,is_delay,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Origin,OriginState,Dest,DestState,CRSDepTime,Cancelled,Diverted,Distance,DistanceGroup,ArrDelay,ArrDelayMinutes,AirTime
0,1.0,2014,1,1,1,3,2014-01-01,UA,LAX,CA,ORD,IL,900,0.0,0.0,1744.0,7,43.0,43.0,218.0
1,0.0,2014,1,1,1,3,2014-01-01,AA,IAH,TX,DFW,TX,1750,0.0,0.0,224.0,1,2.0,2.0,50.0
2,1.0,2014,1,1,1,3,2014-01-01,AA,LAX,CA,ORD,IL,1240,0.0,0.0,1744.0,7,26.0,26.0,220.0
3,1.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,LAX,CA,1905,0.0,0.0,1235.0,5,159.0,159.0,169.0
4,0.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,CLT,NC,1115,0.0,0.0,936.0,4,-13.0,0.0,108.0


In [3]:
df = pd.read_csv(
    "merged_output.csv",
    engine="python",
    on_bad_lines="skip"
)

df.head()


,is_delay,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Origin,OriginState,Dest,DestState,CRSDepTime,Cancelled,Diverted,Distance,DistanceGroup,ArrDelay,ArrDelayMinutes,AirTime
0,1.0,2014,1,1,1,3,2014-01-01,UA,LAX,CA,ORD,IL,900,0.0,0.0,1744.0,7,43.0,43.0,218.0
1,0.0,2014,1,1,1,3,2014-01-01,AA,IAH,TX,DFW,TX,1750,0.0,0.0,224.0,1,2.0,2.0,50.0
2,1.0,2014,1,1,1,3,2014-01-01,AA,LAX,CA,ORD,IL,1240,0.0,0.0,1744.0,7,26.0,26.0,220.0
3,1.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,LAX,CA,1905,0.0,0.0,1235.0,5,159.0,159.0,169.0
4,0.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,CLT,NC,1115,0.0,0.0,936.0,4,-13.0,0.0,108.0


In [50]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:\n", df.columns.tolist())

print("\nInfo:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())


Shape (rows, columns): (1635587, 20)

Column names:
 ['is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'Origin', 'OriginState', 'Dest', 'DestState', 'CRSDepTime', 'Cancelled', 'Diverted', 'Distance', 'DistanceGroup', 'ArrDelay', 'ArrDelayMinutes', 'AirTime']

Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1635587 entries, 0 to 1635589
Data columns (total 20 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   is_delay           1635587 non-null  float64
 1   Year               1635587 non-null  int64  
 2   Quarter            1635587 non-null  int64  
 3   Month              1635587 non-null  int64  
 4   DayofMonth         1635587 non-null  int64  
 5   DayOfWeek          1635587 non-null  int64  
 6   FlightDate         1635587 non-null  object 
 7   Reporting_Airline  1635587 non-null  object 
 8   Origin             1635587 non-null  object 
 9   OriginState       

In [52]:
# Drop columns that look like index columns
to_drop = [col for col in df.columns if col.lower().startswith("unnamed")]
df = df.drop(columns=to_drop, errors='ignore')

print("After dropping unnamed columns:", df.shape)


After dropping unnamed columns: (1635587, 20)


In [66]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"Removed {before - after} duplicate rows")
print("Current shape:", df.shape)


Removed 0 duplicate rows
Current shape: (1635587, 21)


In [67]:
# Check if ARR_DELAY exists
'ARR_DELAY' in df.columns


False

In [70]:
df['Delayed'] = (df['ArrDelayMinutes'] > 15).astype(int)
print("Created 'Delayed' from ArrDelayMinutes. Counts:")
print(df['Delayed'].value_counts())


Created 'Delayed' from ArrDelayMinutes. Counts:
Delayed
0    1304621
1     330966
Name: count, dtype: int64


In [76]:
# quick checks
print(df[['ArrDelay','ArrDelayMinutes','is_delay','Delayed']].describe(include='all'))
print("\nSample rows with delays:")
display(df[df['Delayed']==1].head(5))


           ArrDelay  ArrDelayMinutes      is_delay       Delayed
count  1.635587e+06     1.635587e+06  1.635587e+06  1.635587e+06
mean   6.015965e+00     1.372092e+01  2.099136e-01  2.023530e-01
std    4.298097e+01     3.943790e+01  4.072469e-01  4.017541e-01
min   -8.700000e+01     0.000000e+00  0.000000e+00  0.000000e+00
25%   -1.400000e+01     0.000000e+00  0.000000e+00  0.000000e+00
50%   -4.000000e+00     0.000000e+00  0.000000e+00  0.000000e+00
75%    1.000000e+01     1.000000e+01  0.000000e+00  0.000000e+00
max    2.142000e+03     2.142000e+03  1.000000e+00  1.000000e+00

Sample rows with delays:


,is_delay,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Origin,OriginState,...,DestState,CRSDepTime,Cancelled,Diverted,Distance,DistanceGroup,ArrDelay,ArrDelayMinutes,AirTime,Delayed
0,1.0,2014,1,1,1,3,2014-01-01,UA,LAX,CA,...,IL,900,0.0,0.0,1744.0,7,43.0,43.0,218.0,1
2,1.0,2014,1,1,1,3,2014-01-01,AA,LAX,CA,...,IL,1240,0.0,0.0,1744.0,7,26.0,26.0,220.0,1
3,1.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,...,CA,1905,0.0,0.0,1235.0,5,159.0,159.0,169.0,1
6,1.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,...,CO,1145,0.0,0.0,641.0,3,24.0,24.0,96.0,1
7,1.0,2014,1,1,1,3,2014-01-01,AA,DEN,CO,...,TX,1330,0.0,0.0,641.0,3,26.0,26.0,77.0,1


In [78]:
print(df['Delayed'].value_counts(normalize=True))


Delayed
0    0.797647
1    0.202353
Name: proportion, dtype: float64


In [80]:
df.to_csv("cleaned_with_delayed.csv", index=False)
print("Saved cleaned_with_delayed.csv")


Saved cleaned_with_delayed.csv


In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib


In [84]:
df = pd.read_csv("cleaned_with_delayed.csv")
print("Shape:", df.shape)
df.head()


Shape: (1635587, 21)


,is_delay,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Origin,OriginState,...,DestState,CRSDepTime,Cancelled,Diverted,Distance,DistanceGroup,ArrDelay,ArrDelayMinutes,AirTime,Delayed
0,1.0,2014,1,1,1,3,2014-01-01,UA,LAX,CA,...,IL,900,0.0,0.0,1744.0,7,43.0,43.0,218.0,1
1,0.0,2014,1,1,1,3,2014-01-01,AA,IAH,TX,...,TX,1750,0.0,0.0,224.0,1,2.0,2.0,50.0,0
2,1.0,2014,1,1,1,3,2014-01-01,AA,LAX,CA,...,IL,1240,0.0,0.0,1744.0,7,26.0,26.0,220.0,1
3,1.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,...,CA,1905,0.0,0.0,1235.0,5,159.0,159.0,169.0,1
4,0.0,2014,1,1,1,3,2014-01-01,AA,DFW,TX,...,NC,1115,0.0,0.0,936.0,4,-13.0,0.0,108.0,0


In [86]:
# Choose features (edit if you want to add/remove)
num_feats = ['Month', 'DayOfWeek', 'Distance']
time_col = 'CRSDepTime'     # appears as HHMM integer
cat_feats = ['Reporting_Airline', 'Origin', 'Dest']
target = 'Delayed'

# Ensure these columns exist
for c in num_feats + [time_col] + cat_feats + [target]:
    if c not in df.columns:
        raise KeyError(f"Missing column: {c}")

# Convert CRSDepTime HHMM -> minutes since midnight
def time_to_minutes(x):
    try:
        x = int(x)
        h = x // 100
        m = x % 100
        return h*60 + m
    except:
        return np.nan

df['CRSDep_MIN'] = df[time_col].apply(time_to_minutes)

# Build feature dataframe
features = num_feats + ['CRSDep_MIN'] + cat_feats
print("Features used:", features)
df_features = df[features + [target]].copy()
df_features.head()


Features used: ['Month', 'DayOfWeek', 'Distance', 'CRSDep_MIN', 'Reporting_Airline', 'Origin', 'Dest']


,Month,DayOfWeek,Distance,CRSDep_MIN,Reporting_Airline,Origin,Dest,Delayed
0,1,3,1744.0,540,UA,LAX,ORD,1
1,1,3,224.0,1070,AA,IAH,DFW,0
2,1,3,1744.0,760,AA,LAX,ORD,1
3,1,3,1235.0,1145,AA,DFW,LAX,1
4,1,3,936.0,675,AA,DFW,CLT,0


In [93]:
SAMPLE = True    # <- set False to train on full dataset
SAMPLE_SIZE = 200000  # number of rows if sampling

if SAMPLE:
    # stratified sampling by target to keep class balance
    frac = SAMPLE_SIZE / len(df_features)
    df_sample = df_features.groupby(target, group_keys=False).apply(
        lambda x: x.sample(frac=min(1, frac), random_state=42)
    ).reset_index(drop=True)
    print("Sample shape:", df_sample.shape)
else:
    df_sample = df_features.copy()
    print("Using full data shape:", df_sample.shape)


Sample shape: (200000, 8)


C:\Users\krith\AppData\Local\Temp\ipykernel_20612\1276230043.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df_features.groupby(target, group_keys=False).apply(


In [95]:
TOP_K = 30   # max categories per categorical column to keep (reduce sparsity)

def top_k_onehot(df, col, top_k=TOP_K):
    top = df[col].value_counts().nlargest(top_k).index
    df[col] = df[col].where(df[col].isin(top), other="Other")
    return df

df_enc = df_sample.copy()
for c in cat_feats:
    df_enc = top_k_onehot(df_enc, c, TOP_K)

# Now get dummies
df_enc = pd.get_dummies(df_enc, columns=cat_feats, drop_first=True)
print("After encoding:", df_enc.shape)


After encoding: (200000, 25)


In [97]:
X = df_enc.drop(columns=[target])
y = df_enc[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train class distribution:\n", y_train.value_counts(normalize=True))


Train shape: (160000, 24) Test shape: (40000, 24)
Train class distribution:
 Delayed
0    0.797644
1    0.202356
Name: proportion, dtype: float64


In [99]:
dt = DecisionTreeClassifier(max_depth=12, min_samples_split=100, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
print("Decision Tree accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


Decision Tree accuracy: 0.601225
              precision    recall  f1-score   support

           0       0.85      0.60      0.71     31906
           1       0.27      0.59      0.37      8094

    accuracy                           0.60     40000
   macro avg       0.56      0.60      0.54     40000
weighted avg       0.74      0.60      0.64     40000



In [101]:
rf = RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("Random Forest accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest accuracy: 0.6801
              precision    recall  f1-score   support

           0       0.84      0.74      0.79     31906
           1       0.31      0.46      0.37      8094

    accuracy                           0.68     40000
   macro avg       0.57      0.60      0.58     40000
weighted avg       0.73      0.68      0.70     40000



In [103]:
cm = confusion_matrix(y_test, y_pred_rf)
print("Confusion matrix (rows=true, cols=pred):\n", cm)


Confusion matrix (rows=true, cols=pred):
 [[23516  8390]
 [ 4406  3688]]


In [105]:
model_filename = "FlightDelayPredictionModel.pkl"
joblib.dump(rf, model_filename)
print("Saved model to", model_filename)


Saved model to FlightDelayPredictionModel.pkl


In [107]:
feature_columns = X.columns.tolist()
pd.Series(feature_columns).to_csv("model_feature_columns.csv", index=False)
print("Saved feature column list (length):", len(feature_columns))


Saved feature column list (length): 24
